# 조선 섹터 퀀트 케이스스터디
## Part A. 조선주는 언제 '테마'로 움직이나 — 섹터 동조화 측정
## Part B. 수주 공시는 뉴스인가 — 수주 공시 이벤트 스터디

**사용법 (Google Colab 기준)**
1. [OpenDART](https://opendart.fss.or.kr) 가입 → 인증키 신청 (무료, 즉시 발급)
2. 아래 설정 셀의 `DART_API_KEY`에 키 입력
3. `런타임 → 모두 실행`

| 가설 | 내용 | 검정 |
|---|---|---|
| A1 | 조선주 간 초과수익 상관(테마 강도)은 시기별로 크게 변하고, 특정 이벤트에 급등한다 | 60일 롤링 평균 상관 |
| A2 | 테마 강도가 높은 구간 이후 섹터 초과수익은 반전된다 | 레짐별 선행 수익률, Newey-West t |
| H1 | 수주 공시일 [0,+1]의 초과수익은 작다 (이미 반영) | 시장모형 CAR |
| H2 | 공시 전 [-10,-1]에 유의한 초과수익이 있다 (선반영·정보유출) | 시장모형 CAR |
| H3 | 테마 강도가 높을 때 개별 공시 반응은 약하고, 경쟁사로의 스필오버는 크다 | 레짐별 CAR, 경쟁사 CAR |

In [ ]:
# %pip = 현재 커널의 python에 설치 (!pip는 PATH상의 다른 python에 설치될 수 있음)
%pip install -q finance-datareader pykrx opendartreader statsmodels

In [ ]:
import os, time, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.dpi"] = 110

DART_API_KEY = os.environ["DART_API_KEY"]    # 환경변수에서 읽음 (키 문자열을 노트북·출력에 남기지 않음)
assert DART_API_KEY.strip(), "환경변수 DART_API_KEY가 비어 있습니다"
START = "2015-01-01"
END   = pd.Timestamp.today().strftime("%Y-%m-%d")
RES_DIR = "results"                          # Part D·E 그래프·표 저장 폴더
os.makedirs(RES_DIR, exist_ok=True)

# 조선 유니버스  종목코드: (그래프용 영문 라벨, 그룹)
# 합병·상장폐지된 종목은 데이터가 있는 기간까지만 자동 사용됨
UNIVERSE = {
    "009540": ("HD KSOE(Hold)",  "HD"),
    "329180": ("HD HHI",         "HD"),
    "010620": ("HD Mipo",        "HD"),
    "010140": ("Samsung HI",     "SHI"),
    "042660": ("Hanwha Ocean",   "HANWHA"),
    "097230": ("HJ Shipbuilding","HJ"),
}
NAME  = {k: v[0] for k, v in UNIVERSE.items()}
GROUP = {k: v[1] for k, v in UNIVERSE.items()}

# 지주사는 자회사와 기계적으로 상관이 높아 동조화 계산에서 제외
CORR_EXCLUDE = ["009540"]
# HJ중공업은 건설부문 공급계약도 같은 이름으로 공시되므로 이벤트 스터디 기본값에서 제외
EVENT_TICKERS = ["009540", "329180", "010620", "010140", "042660"]

ROLL_WIN    = 60          # 동조화 롤링 윈도우(거래일)
EST_WIN     = (-250, -30) # 시장모형 추정구간
EVT_WIN     = (-10, 10)   # 이벤트 윈도우
MIN_EST_OBS = 120
REL = np.arange(EVT_WIN[0], EVT_WIN[1] + 1)

---
## Step 0. 주가 데이터
pykrx(수정주가)를 우선 쓰고, 실패하면 FinanceDataReader로 대체합니다.
거래정지일(거래량 0)과 일간 ±30% 초과 수익률(가격제한폭 밖 = 감자·데이터 오류)은 제거합니다.

In [ ]:
def _load_one(ticker, start, end):
    s, e = start.replace("-", ""), end.replace("-", "")
    try:
        from pykrx import stock
        df = stock.get_market_ohlcv(s, e, ticker)          # adjusted=True 기본
        if df is not None and not df.empty:
            return df.rename(columns={"종가": "Close", "거래량": "Volume"})[["Close", "Volume"]]
    except Exception as ex:
        print(f"  pykrx 실패({ticker}): {ex}")
    try:
        import FinanceDataReader as fdr
        df = fdr.DataReader(ticker, start, end)
        if df is not None and not df.empty:
            print(f"  FDR 사용({ticker})")
            return df[["Close", "Volume"]]
    except Exception as ex:
        print(f"  FDR 실패({ticker}): {ex}")
    return None

def _load_kospi(start, end):
    try:
        from pykrx import stock
        df = stock.get_index_ohlcv(start.replace("-", ""), end.replace("-", ""), "1001")
        if df is not None and not df.empty:
            return df["종가"].rename("KOSPI")
    except Exception as ex:
        print(f"  pykrx 지수 실패: {ex}")
    import FinanceDataReader as fdr
    return fdr.DataReader("KS11", start, end)["Close"].rename("KOSPI")

def load_all(universe, start, end):
    close, vol = {}, {}
    for t in universe:
        print(f"loading {t} {NAME[t]}")
        df = _load_one(t, start, end)
        if df is None:
            print(f"  -> {t} 제외")
            continue
        close[t], vol[t] = df["Close"], df["Volume"]
        time.sleep(0.3)
    kospi = _load_kospi(start, end)
    kospi.index = pd.to_datetime(kospi.index)
    close = pd.DataFrame(close); close.index = pd.to_datetime(close.index)
    vol   = pd.DataFrame(vol);   vol.index   = pd.to_datetime(vol.index)
    return close.reindex(kospi.index), vol.reindex(kospi.index), kospi

def to_returns(close, vol, kospi, cap=0.30, tol=0.005):
    ret = close.pct_change(fill_method=None)
    ret = ret.mask((vol == 0) | vol.isna())
    # 정확히 ±30%인 상·하한가는 정상 관측치 → 부동소수점(1.3-1=0.30000000000000004)·수정주가 반올림 오차만큼 허용
    bad = ret.abs() > cap + tol
    if int(bad.values.sum()):
        print(f"|일간수익률| > {cap + tol:.1%} 관측치 {int(bad.values.sum())}개 NaN 처리")
    ret = ret.mask(bad)
    mkt = kospi.pct_change(fill_method=None)
    return ret, mkt

In [ ]:
close, vol, kospi = load_all(UNIVERSE, START, END)
ret, mkt = to_returns(close, vol, kospi)
print("\n종목별 유효 수익률 관측치:")
print(ret.notna().sum().rename(NAME))

---
# Part A. 섹터 동조화 (테마 강도)
**테마 강도** = 조선주들의 *시장조정수익률*(종목수익률 − KOSPI) 사이 60일 평균 쌍별 상관계수  
시장 전체가 같이 움직여서 생기는 상관은 빼고, "조선이라서" 같이 움직이는 부분만 남기는 것이 핵심.

레짐 구분은 **룩어헤드 방지**를 위해 그날까지의 과거 분포 대비 백분위(expanding rank)로 합니다.
상위 20% = High, 하위 20% = Low.

In [ ]:
def rolling_avg_corr(x, window=60, min_assets=3, min_frac=0.8):
    vals = np.full(len(x), np.nan)
    for i in range(window - 1, len(x)):
        sub = x.iloc[i - window + 1:i + 1]
        sub = sub.loc[:, sub.notna().sum() >= int(window * min_frac)]
        if sub.shape[1] < min_assets:
            continue
        c = sub.corr().values
        iu = np.triu_indices(c.shape[0], 1)
        vals[i] = np.nanmean(c[iu])
    return pd.Series(vals, index=x.index)

corr_cols = [c for c in ret.columns if c not in CORR_EXCLUDE]
ex_ret  = ret[corr_cols].sub(mkt, axis=0)                  # 시장조정수익률
theme     = rolling_avg_corr(ex_ret,       ROLL_WIN).rename("theme (market-adj)")
theme_raw = rolling_avg_corr(ret[corr_cols], ROLL_WIN).rename("raw corr")

theme_rank = theme.expanding(min_periods=250).rank(pct=True)
def to_regime(r):
    return pd.cut(r, [0, 0.2, 0.8, 1.0], labels=["Low", "Mid", "High"], include_lowest=True)
regime = to_regime(theme_rank)

sector_ex = ex_ret.mean(axis=1, skipna=True)               # 동일가중 섹터 초과수익
print(theme.describe())
print("\n레짐 분포:\n", regime.value_counts())

In [ ]:
def top_peaks(s, k=6, gap=60):
    s = s.dropna().copy(); picks = []
    while len(picks) < k and len(s):
        d = s.idxmax(); picks.append((d.date(), s[d]))
        p = s.index.get_loc(d)
        s = s.drop(s.index[max(0, p - gap):p + gap + 1])
    return pd.DataFrame(picks, columns=["date", "theme_corr"])

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw={"height_ratios": [1.3, 1]})
ax[0].plot(theme.index, theme, lw=1.2, label="Theme strength (market-adjusted corr)")
ax[0].plot(theme_raw.index, theme_raw, lw=0.8, alpha=0.5, label="Raw return corr")
hi = (regime == "High")
ax[0].fill_between(theme.index, 0, 1, where=hi.values, transform=ax[0].get_xaxis_transform(),
                   color="tab:red", alpha=0.08, label="High regime")
ax[0].set_ylabel("avg pairwise corr"); ax[0].legend(loc="upper left", fontsize=8)
ax[0].set_title(f"Shipbuilding sector co-movement ({ROLL_WIN}d rolling)")
ax[1].plot(sector_ex.index, sector_ex.fillna(0).cumsum() * 100, color="k", lw=1)
ax[1].set_ylabel("cum. excess ret vs KOSPI (%)")
ax[1].set_title("Equal-weight shipbuilding excess return (cumulative)")
plt.tight_layout(); plt.savefig("fig_A1_theme.png", bbox_inches="tight"); plt.show()

peaks = top_peaks(theme)
print("테마 강도 상위 시점 → 이 날짜 전후 뉴스를 찾아 슬라이드에 매칭하세요")
display(peaks)

### A2. 테마 강도가 높은 뒤에는 반전되는가
t일 레짐 기준으로 t+1 ~ t+h 섹터 초과수익 합을 보고, 겹치는 보유기간 때문에 Newey-West(lag = h) 표준오차를 씁니다.

In [ ]:
def fwd_sum(s, h):
    return s.rolling(h, min_periods=int(h * 0.8)).sum().shift(-h)

def nw_mean(y, lags):
    y = y.dropna()
    if len(y) < 30:
        return np.nan, np.nan, len(y)
    res = sm.OLS(y.values, np.ones(len(y))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return res.params[0], res.tvalues[0], len(y)

def nw_diff(y, is_high, lags):
    d = pd.concat([y, is_high.astype(float)], axis=1).dropna()
    X = sm.add_constant(d.iloc[:, 1].values)
    res = sm.OLS(d.iloc[:, 0].values, X).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return res.params[1], res.tvalues[1]

rows = []
for h in [5, 20, 60]:
    f = fwd_sum(sector_ex, h)
    for g in ["Low", "Mid", "High"]:
        m, t, n = nw_mean(f[regime == g], h)
        rows.append({"horizon": f"{h}d", "regime": g, "mean fwd excess(%)": m * 100, "NW t": t, "N(days)": n})
    sub = regime.isin(["Low", "High"])
    d, t = nw_diff(f[sub], (regime[sub] == "High"), h)
    rows.append({"horizon": f"{h}d", "regime": "High − Low", "mean fwd excess(%)": d * 100, "NW t": t, "N(days)": np.nan})
tblA2 = pd.DataFrame(rows).set_index(["horizon", "regime"])
display(tblA2)
print("해석: High − Low가 음수이고 |t| > 2 → 테마 과열 후 반전(A2 지지)")

---
# Part B. 수주 공시 이벤트 스터디
DART '단일판매ㆍ공급계약체결' 공시 (정정·해지 공시 제외).  
- 공시일이 휴장일이면 다음 거래일을 t=0으로, 장 마감 후 공시를 고려해 공시 반응은 [0,+1]로 봅니다.  
- 같은 종목·같은 날 여러 건이면 한 이벤트로 묶습니다.  
- **Clean 이벤트**: 같은 종목의 앞뒤 10거래일 안에 다른 수주 공시가 없는 것. 조선사는 수주가 몰려 나오므로 윈도우 오염을 막기 위한 필터입니다.

In [ ]:
import OpenDartReader
dart = OpenDartReader(DART_API_KEY)

def fetch_contracts(ticker):
    df = None
    for kind in ["I", ""]:
        try:
            df = dart.list(ticker, start=START, end=END, kind=kind, final=False)
            if df is not None and not df.empty:
                break
        except Exception as ex:
            print(f"  DART 실패({ticker}, kind={kind!r}): {ex}")
    if df is None or df.empty:
        return pd.DataFrame()
    nm = df["report_nm"].astype(str)
    m = (nm.str.contains("단일판매") & nm.str.contains("공급계약") & nm.str.contains("체결")
         & ~nm.str.contains("정정") & ~nm.str.contains("해지"))
    out = df.loc[m, ["rcept_no", "rcept_dt", "report_nm"]].copy()
    out["ticker"] = ticker
    return out

raw = []
for t in EVENT_TICKERS:
    if t not in ret.columns:
        continue
    r = fetch_contracts(t)
    print(f"{t} {NAME[t]}: 수주 공시 {len(r)}건")
    raw.append(r); time.sleep(0.5)
raw = pd.concat(raw, ignore_index=True)
raw["rcept_dt"] = pd.to_datetime(raw["rcept_dt"].astype(str))
raw.head()

In [ ]:
def build_events(raw, cal, gap=None):
    gap = gap or abs(EVT_WIN[0])
    raw = raw.copy()
    p = cal.searchsorted(raw["rcept_dt"])                # 공시일 이후 첫 거래일
    raw["tdate"] = [cal[i] if i < len(cal) else pd.NaT for i in p]
    ev = (raw.dropna(subset=["tdate"])
             .groupby(["ticker", "tdate"])
             .agg(n_contracts=("rcept_no", "size"), rcept_no=("rcept_no", "first"),
                  report_nm=("report_nm", "first"))
             .reset_index())
    ev["pos"] = [cal.get_loc(d) for d in ev["tdate"]]
    ev = ev.sort_values(["ticker", "tdate"]).reset_index(drop=True)
    prev_gap = ev.groupby("ticker")["pos"].diff()
    next_gap = -ev.groupby("ticker")["pos"].diff(-1)
    ev["clean"] = (prev_gap.isna() | (prev_gap > gap)) & (next_gap.isna() | (next_gap > gap))
    return ev

events = build_events(raw, ret.index)
print(f"이벤트 {len(events)}개 (clean {events['clean'].sum()}개)")
display(pd.crosstab(events["tdate"].dt.year, events["ticker"].map(NAME)))

In [ ]:
def market_model_ar(r, m, pos):
    n = len(r)
    if pos + EST_WIN[0] < 0 or pos + EVT_WIN[1] >= n:
        return None
    y = r.iloc[pos + EST_WIN[0]: pos + EST_WIN[1] + 1].values
    x = m.iloc[pos + EST_WIN[0]: pos + EST_WIN[1] + 1].values
    ok = ~np.isnan(y) & ~np.isnan(x)
    if ok.sum() < MIN_EST_OBS:
        return None
    beta, alpha = np.polyfit(x[ok], y[ok], 1)
    ry = r.iloc[pos + EVT_WIN[0]: pos + EVT_WIN[1] + 1].values
    rx = m.iloc[pos + EVT_WIN[0]: pos + EVT_WIN[1] + 1].values
    ar = ry - (alpha + beta * rx)
    if np.isnan(ar).sum() > 2:          # 이벤트 윈도우에 결측 3일 이상이면 제외
        return None
    return np.nan_to_num(ar)

def run_event_study(ev, ret, mkt):
    paths, keep = [], []
    for i, e in ev.iterrows():
        if e["ticker"] not in ret.columns:
            continue
        ar = market_model_ar(ret[e["ticker"]], mkt, int(e["pos"]))
        if ar is not None:
            paths.append(ar); keep.append(i)
    return pd.DataFrame(paths, index=keep, columns=REL)

def car(AR, a, b):
    return AR.loc[:, a:b].sum(axis=1)

WINDOWS = {"[-10,-1] pre": (-10, -1), "[-5,-1] pre": (-5, -1), "[0,+1] announce": (0, 1),
           "[+2,+10] post": (2, 10), "[-10,+10] total": (-10, 10)}

def car_table(AR):
    rows = []
    for name, (a, b) in WINDOWS.items():
        c = car(AR, a, b)
        t, p = stats.ttest_1samp(c, 0) if len(c) > 2 else (np.nan, np.nan)
        wp = stats.wilcoxon(c).pvalue if len(c) > 10 else np.nan
        rows.append({"window": name, "mean CAR(%)": c.mean() * 100, "median(%)": c.median() * 100,
                     "t": t, "p": p, "wilcoxon p": wp, "% positive": (c > 0).mean() * 100, "N": len(c)})
    return pd.DataFrame(rows).set_index("window")

def plot_car(AR_dict, title, fname):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    for label, AR in AR_dict.items():
        if len(AR) < 3:
            continue
        cp = AR.cumsum(axis=1)
        mu = cp.mean() * 100
        se = cp.std() / np.sqrt(len(cp)) * 100
        ax.plot(REL, mu, marker="o", ms=3, label=f"{label} (N={len(cp)})")
        ax.fill_between(REL, mu - 1.96 * se, mu + 1.96 * se, alpha=0.15)
    ax.axvline(0, color="k", lw=0.8, ls="--"); ax.axhline(0, color="grey", lw=0.6)
    ax.set_xlabel("trading days relative to disclosure"); ax.set_ylabel("mean CAR (%)")
    ax.set_title(title); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(fname, bbox_inches="tight"); plt.show()

### H1 · H2 — 공시일 반응 vs 사전 반응

In [ ]:
AR_all = run_event_study(events, ret, mkt)
ev_ok  = events.loc[AR_all.index].copy()
AR_clean = AR_all.loc[ev_ok.index[ev_ok["clean"]]]

print("■ 전체 이벤트"); display(car_table(AR_all))
print("■ Clean 이벤트 (앞뒤 10거래일 내 다른 공시 없음)"); display(car_table(AR_clean))
plot_car({"All events": AR_all, "Clean events": AR_clean},
         "Shipbuilding order disclosures: mean CAR (95% CI)", "fig_B1_car.png")

firm = pd.DataFrame({
    "pre[-10,-1](%)": car(AR_all, -10, -1) * 100,
    "ann[0,+1](%)":   car(AR_all, 0, 1) * 100,
    "firm": ev_ok["ticker"].map(NAME)}).groupby("firm").agg(["mean", "count"])
print("■ 종목별"); display(firm)
print("해석: [0,+1]이 작고 [-10,-1]이 유의 → 수주 정보가 공시 전에 이미 가격에 반영(H1·H2 지지)")

### H3 — 테마 강도에 따라 반응이 달라지는가 + 경쟁사 스필오버
- 레짐은 **공시 전날(t−1)** 기준 → 공시 자체가 상관을 올리는 역인과 방지  
- 스필오버: 공시 기업과 **다른 그룹**의 조선사 CAR 평균 (HD그룹 내 지주-자회사는 제외).  
  같은 날 같은 그룹 공시(예: 지주사+자회사 동시 공시)는 한 번만 셉니다.

In [ ]:
rank_lag = theme_rank.shift(1)
ev_ok["regime"] = to_regime(rank_lag.reindex(ev_ok["tdate"])).values
ev_ok["own_pre"] = car(AR_all, -10, -1)
ev_ok["own_ann"] = car(AR_all, 0, 1)

# --- 경쟁사 스필오버 ---
src = ev_ok.assign(grp=ev_ok["ticker"].map(GROUP)).drop_duplicates(["grp", "tdate"])
peer_rows = []
for i, e in src.iterrows():
    for p in ret.columns:
        if p in CORR_EXCLUDE or GROUP.get(p) == GROUP[e["ticker"]]:
            continue
        peer_rows.append({"src": i, "ticker": p, "tdate": e["tdate"], "pos": e["pos"]})
peer_ev = pd.DataFrame(peer_rows)
AR_peer = run_event_study(peer_ev, ret, mkt)
peer_ev = peer_ev.loc[AR_peer.index]
peer = pd.DataFrame({"src": peer_ev["src"].values,
                     "peer_pre": car(AR_peer, -10, -1).values,
                     "peer_ann": car(AR_peer, 0, 1).values}).groupby("src").mean()
src = src.join(peer, how="inner")

def regime_table(df, cols):
    out = df.groupby("regime", observed=False)[cols].agg(["mean", "count"])
    for c in cols:
        out[(c, "mean")] *= 100
    return out

print("■ 레짐별 자사 CAR (%)"); display(regime_table(ev_ok, ["own_pre", "own_ann"]))
print("■ 레짐별 경쟁사 CAR (%)"); display(regime_table(src, ["peer_pre", "peer_ann"]))

def welch(df, col):
    a = df.loc[df["regime"] == "High", col].dropna(); b = df.loc[df["regime"] == "Low", col].dropna()
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan
    t, p = stats.ttest_ind(a, b, equal_var=False)
    return (a.mean() - b.mean()) * 100, p

rows = []
for df, col in [(ev_ok, "own_ann"), (ev_ok, "own_pre"), (src, "peer_ann"), (src, "peer_pre")]:
    d, p = welch(df, col); rows.append({"variable": col, "High − Low (%p)": d, "Welch p": p})
display(pd.DataFrame(rows).set_index("variable"))

labels = ["Low", "Mid", "High"]
own = [ev_ok.loc[ev_ok["regime"] == g, "own_ann"].mean() * 100 for g in labels]
prr = [src.loc[src["regime"] == g, "peer_ann"].mean() * 100 for g in labels]
x = np.arange(3); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - w/2, own, w, label="Announcing firm CAR[0,+1]")
ax.bar(x + w/2, prr, w, label="Peer firms CAR[0,+1]")
ax.set_xticks(x); ax.set_xticklabels([f"{g} theme" for g in labels]); ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("mean CAR (%)"); ax.set_title("Order disclosure reaction by theme regime"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_B2_regime.png", bbox_inches="tight"); plt.show()
print("해석: High에서 자사 반응↓·경쟁사 반응↑ → 테마 구간엔 개별 수주가 '섹터 뉴스'로 소비됨(H3 지지)")

### (선택) 계약 규모 효과 — 매출액 대비 계약금액
공시 원문에서 '매출액 대비(%)'를 파싱합니다. 공시 양식이 조금씩 달라 일부는 실패(NaN)할 수 있고, 이벤트 수만큼 API를 호출하므로 몇 분 걸립니다.

In [ ]:
RUN_SIZE = False     # True로 바꾸면 실행

def parse_sales_ratio(rcept_no):
    try:
        xml = dart.document(rcept_no)
    except Exception:
        return np.nan
    txt = re.sub(r"<[^>]+>", " ", str(xml)); txt = re.sub(r"\s+", " ", txt)
    m = re.search(r"매출액\s*대비\s*\(?\s*%?\s*\)?\s*([0-9]+(?:\.[0-9]+)?)", txt)
    return float(m.group(1)) if m else np.nan

if RUN_SIZE:
    ratios = []
    for rn in ev_ok["rcept_no"]:
        ratios.append(parse_sales_ratio(rn)); time.sleep(0.15)
    ev_ok["sales_ratio"] = ratios
    print("파싱 성공률:", ev_ok["sales_ratio"].notna().mean().round(2))
    med = ev_ok["sales_ratio"].median()
    ev_ok["size_grp"] = np.where(ev_ok["sales_ratio"] >= med, "Large", "Small")
    ev_ok.loc[ev_ok["sales_ratio"].isna(), "size_grp"] = np.nan
    display(ev_ok.groupby("size_grp")[["own_pre", "own_ann"]].agg(["mean", "count"]))

---
## 결과 저장
`events_with_car.csv`에는 이벤트별 CAR이 들어 있습니다. 사전 CAR이 큰 이벤트 몇 개를 골라 **공시 전 언론 보도가 있었는지** 직접 확인하면 H2를 사례로 보강할 수 있습니다.

In [ ]:
out = ev_ok.assign(firm=ev_ok["ticker"].map(NAME))[
    ["firm", "tdate", "report_nm", "n_contracts", "clean", "regime", "own_pre", "own_ann"]]
out.to_csv("events_with_car.csv", index=False, encoding="utf-8-sig")
tblA2.to_csv("A2_regime_forward.csv", encoding="utf-8-sig")
print("저장 완료: events_with_car.csv, A2_regime_forward.csv, fig_A1_theme.png, fig_B1_car.png, fig_B2_regime.png")
print("\n사전 CAR 상위 10개 이벤트 (뉴스 확인용):")
display(out.sort_values("own_pre", ascending=False).head(10))

---
# Part C. 매매전략 — 테마 과열 회피 & 수주 모멘텀
Part A·B에서 찾은 현상을 실제 포지션으로 옮깁니다. **공매도 없이 롱온리**로만 구성해 개인도 실행 가능한 형태로 만들고,
모든 신호는 t일 종가까지의 정보로 만들어 **t+1일 수익률**에 적용합니다 (룩어헤드 차단).

| 전략 | 신호 | 논리 |
|---|---|---|
| S1 테마 과열 회피 | 테마 강도 백분위 ≥ 0.8이면 비중 0 | 과열 구간 이후 반전(A2) |
| S2 수주 모멘텀 | 최근 60거래일 수주 공시 건수가 과거 대비 상위면 비중 1 | 수주 증가 = 펀더멘털 개선 |
| S3 결합 | S1·S2 모두 만족할 때만 보유 | 두 신호의 교집합 |
| B&H | 항상 보유 | 벤치마크 |

비용은 매수 수수료, 매도 수수료·거래세를 회전율에 곱해 차감합니다. **세율은 최신 기준으로 확인해서 넣으세요.**

In [ ]:
COST_BUY  = 0.00015              # 매수 수수료
COST_SELL = 0.00015 + 0.0015     # 매도 수수료 + 거래세 (※ 최신 세율 확인 후 수정)
IS_END    = "2020-12-31"         # 여기까지 In-sample, 이후 Out-of-sample

sector_ret = ret[corr_cols].mean(axis=1, skipna=True).fillna(0)   # 동일가중 조선 섹터
bench      = mkt.fillna(0)

def backtest(weight, asset_ret, cost_buy=COST_BUY, cost_sell=COST_SELL):
    w = weight.shift(1).fillna(0).clip(0, 1)          # t 신호 → t+1 적용
    dw = w.diff().fillna(w)
    cost = dw.clip(lower=0) * cost_buy + (-dw).clip(lower=0) * cost_sell
    gross = w * asset_ret
    return pd.DataFrame({"w": w, "gross": gross, "net": gross - cost, "cost": cost})

def metrics(r, name="", rf=0.0):
    r = r.dropna()
    if len(r) < 60:
        return {}
    cum = (1 + r).prod()
    yrs = len(r) / 252
    cagr = cum ** (1 / yrs) - 1
    vol = r.std() * np.sqrt(252)
    eq = (1 + r).cumprod()
    mdd = (eq / eq.cummax() - 1).min()
    return {"strategy": name, "CAGR(%)": cagr * 100, "Vol(%)": vol * 100,
            "Sharpe": (cagr - rf) / vol if vol > 0 else np.nan, "MDD(%)": mdd * 100,
            "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
            "Hit(%)": (r > 0).mean() * 100, "Days": len(r)}

In [ ]:
# S1 : 테마 과열이 아니면 보유
sig1 = (theme_rank < 0.8).astype(float)
sig1[theme_rank.isna()] = 1.0          # 레짐 판정 전 구간은 단순 보유

# S2 : 수주 공시 강도 (최근 60거래일 건수의 과거 대비 백분위)
cnt = pd.Series(0.0, index=ret.index)
vc = events.groupby("tdate")["n_contracts"].sum()
cnt.loc[cnt.index.intersection(vc.index)] = vc.reindex(cnt.index.intersection(vc.index)).values
flow = cnt.rolling(60).sum()
flow_rank = flow.expanding(min_periods=250).rank(pct=True)
sig2 = (flow_rank >= 0.4).astype(float)
sig2[flow_rank.isna()] = 1.0

sig3 = ((sig1 > 0) & (sig2 > 0)).astype(float)
bh   = pd.Series(1.0, index=ret.index)

STRATS = {"B&H (sector)": bh, "S1 theme-filter": sig1, "S2 order-momentum": sig2, "S3 combined": sig3}
print("평균 투자 비중:", {k: round(v.shift(1).fillna(0).mean(), 2) for k, v in STRATS.items()})

In [ ]:
res, rows = {}, []
for name, sig in STRATS.items():
    bt = backtest(sig, sector_ret)
    res[name] = bt
    m = metrics(bt["net"], name)
    m["turnover/yr"] = bt["w"].diff().abs().sum() / (len(bt) / 252)
    m["cost drag(%p)"] = (metrics(bt["gross"], name)["CAGR(%)"] - m["CAGR(%)"])
    rows.append(m)
rows.append(metrics(bench, "KOSPI"))
display(pd.DataFrame(rows).set_index("strategy"))

fig, ax = plt.subplots(figsize=(10, 5))
for name, bt in res.items():
    ax.plot(bt.index, (1 + bt["net"]).cumprod(), lw=1.2, label=name)
ax.plot(bench.index, (1 + bench).cumprod(), lw=1, color="grey", ls="--", label="KOSPI")
ax.axvline(pd.Timestamp(IS_END), color="k", lw=0.8, ls=":")
ax.set_yscale("log"); ax.set_ylabel("cumulative (log)"); ax.legend(fontsize=8)
ax.set_title("Shipbuilding sector timing strategies (net of costs)")
plt.tight_layout(); plt.savefig("fig_C1_equity.png", bbox_inches="tight"); plt.show()

### 과최적화 점검 — 기간 분리 & 파라미터 민감도
앞 기간에서 정한 규칙이 뒤 기간에도 통하는지, 임계값을 바꿔도 결론이 유지되는지 봅니다.
여기서 결과가 무너지면 "이 전략은 표본 안에서만 작동했다"가 정직한 결론입니다.

In [ ]:
rows = []
for name, bt in res.items():
    for lbl, sl in [("In-sample", slice(None, IS_END)), ("Out-of-sample", slice(IS_END, None))]:
        m = metrics(bt["net"].loc[sl], name); m["period"] = lbl
        if m: rows.append(m)
display(pd.DataFrame(rows).set_index(["period", "strategy"])[["CAGR(%)", "Sharpe", "MDD(%)"]])

grid = []
for thr in [0.7, 0.75, 0.8, 0.85, 0.9]:
    for cost_mult in [0, 1, 2]:
        s = (theme_rank < thr).astype(float); s[theme_rank.isna()] = 1.0
        bt = backtest(s, sector_ret, COST_BUY * cost_mult, COST_SELL * cost_mult)
        m = metrics(bt["net"], f"thr={thr}")
        grid.append({"threshold": thr, "cost x": cost_mult, "CAGR(%)": m["CAGR(%)"],
                     "Sharpe": m["Sharpe"], "MDD(%)": m["MDD(%)"]})
display(pd.DataFrame(grid).pivot_table(index="threshold", columns="cost x", values=["CAGR(%)", "Sharpe"]))
print("해석: 임계값·비용을 바꿔도 Sharpe 순서가 유지되면 강건, 특정 조합에서만 좋으면 과최적화 의심")

---
# Part D. 페어트레이딩 — 섹터가 아니라 종목 간 상대가격
Part A·C에서 **섹터 방향 타이밍은 실패**했습니다. 그런데 조선주들이 같이 움직인다는 사실(A1)은
반대로 "같이 움직이는 두 종목의 상대가격이 벌어지면 되돌아오는가"라는 질문으로 쓸 수 있습니다.

- 헤지비율과 z-score는 **롤링 윈도우**로만 계산 (전체표본 회귀 = 룩어헤드)
- 진입 |z| ≥ 2, 청산 |z| ≤ 0.5, 신호는 t일 → t+1일 적용
- 비용: 양쪽 다리 모두 수수료·거래세 + 숏 다리 대차수수료(연율)
- **롱숏 버전**(이론)과 **롱온리 스위칭 버전**(개인 실행 가능)을 나란히 비교

In [ ]:
from statsmodels.tsa.stattools import coint, adfuller

BORROW_ANNUAL = 0.03      # 공매도 대차수수료 연율 (기관 기준 가정)
LOOKBACK  = 120           # 헤지비율·z-score 롤링 윈도우
Z_ENTRY, Z_EXIT = 2.0, 0.5

logpx = np.log(close).replace([np.inf, -np.inf], np.nan)

def half_life(spread):
    s = spread.dropna()
    if len(s) < 60: return np.nan
    ds, lag = s.diff().dropna(), s.shift(1).dropna()
    idx = ds.index.intersection(lag.index)
    b = np.polyfit(lag[idx], ds[idx], 1)[0]
    return -np.log(2) / b if b < 0 else np.nan

def screen_pairs(logpx, min_obs=500):
    rows = []
    cols = list(logpx.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a, b = cols[i], cols[j]
            d = logpx[[a, b]].dropna()
            if len(d) < min_obs: continue
            try:
                p = coint(d[a], d[b])[1]
            except Exception:
                continue
            beta = np.polyfit(d[b], d[a], 1)[0]
            sp = d[a] - beta * d[b]
            rows.append({"A": NAME[a], "B": NAME[b], "a": a, "b": b, "obs": len(d),
                         "coint p": p, "ADF p": adfuller(sp)[1],
                         "half-life(d)": half_life(sp),
                         "corr(ret)": ret[[a, b]].dropna().corr().iloc[0, 1]})
    return pd.DataFrame(rows).sort_values("coint p")

pairs = screen_pairs(logpx)
display(pairs.drop(columns=["a", "b"]))
pairs.to_csv(f"{RES_DIR}/D0_coint_screen.csv", index=False, encoding="utf-8-sig")
print("coint p < 0.05 이면 두 종목의 장기 균형관계(공적분)를 기각하지 못함 = 페어 후보")

In [ ]:
def pair_backtest(a, b, lookback=LOOKBACK, z_in=Z_ENTRY, z_out=Z_EXIT,
                  long_only=False, borrow=BORROW_ANNUAL,
                  cost_buy=COST_BUY, cost_sell=COST_SELL):
    d = pd.DataFrame({"la": logpx[a], "lb": logpx[b],
                      "ra": ret[a], "rb": ret[b]}).dropna()
    if len(d) < lookback + 100:
        return None
    # 롤링 헤지비율 (t까지의 정보만)
    cov = d["la"].rolling(lookback).cov(d["lb"])
    var = d["lb"].rolling(lookback).var()
    beta = (cov / var).clip(0.2, 5.0)
    spread = d["la"] - beta * d["lb"]
    z = (spread - spread.rolling(lookback).mean()) / spread.rolling(lookback).std()

    pos = np.zeros(len(d))                      # +1: A롱/B숏, -1: A숏/B롱
    for i in range(1, len(d)):
        p, zi = pos[i - 1], z.iloc[i]
        if np.isnan(zi):
            pos[i] = 0
        elif p == 0:
            pos[i] = 1 if zi <= -z_in else (-1 if zi >= z_in else 0)
        else:
            pos[i] = 0 if abs(zi) <= z_out else p
    pos = pd.Series(pos, index=d.index)
    p1, b1 = pos.shift(1).fillna(0), beta.shift(1).fillna(0)

    if long_only:                                # 싼 쪽만 매수 (숏 없음)
        gross = (p1 > 0) * d["ra"] + (p1 < 0) * d["rb"]
        wa, wb = (p1 > 0).astype(float), (p1 < 0).astype(float)
        borrow_cost = 0.0
    else:
        w = 1 / (1 + b1)
        gross = p1 * w * (d["ra"] - b1 * d["rb"])
        wa, wb = (p1 * w).abs(), (p1 * w * b1).abs()
        # 숏 다리 비중에만 대차수수료: p1>0이면 B 숏(wb), p1<0이면 A 숏(wa)
        short_w = wb.where(p1 > 0, 0.0) + wa.where(p1 < 0, 0.0)
        borrow_cost = short_w * borrow / 252
    trade = wa.diff().abs().fillna(wa) + wb.diff().abs().fillna(wb)
    cost = trade * (cost_buy + cost_sell) / 2 + borrow_cost
    out = pd.DataFrame({"z": z, "pos": pos, "gross": gross, "net": gross - cost})
    out.attrs["trades"] = int((pos.diff().abs() > 0).sum())               # 포지션 변경 횟수(진입+청산)
    out.attrs["round_trips"] = int(((pos != 0) & (pos.shift(1).fillna(0) == 0)).sum())   # 진입 횟수 = 왕복 거래 수
    return out

def pair_report(cands, long_only=False):
    rows, curves = [], {}
    for _, r in cands.iterrows():
        bt = pair_backtest(r["a"], r["b"], long_only=long_only)
        if bt is None: continue
        label = f"{r['A']} / {r['B']}"
        m = metrics(bt["net"], label)
        m["trades"] = bt.attrs["trades"]
        m["round trips"] = bt.attrs["round_trips"]
        m["time in mkt(%)"] = (bt["pos"] != 0).mean() * 100
        rows.append(m); curves[label] = bt["net"]
    return pd.DataFrame(rows).set_index("strategy"), curves

cands = pairs.head(3)
tab_ls, cur_ls = pair_report(cands, long_only=False)
tab_lo, cur_lo = pair_report(cands, long_only=True)
print("■ 롱숏 (이론)"); display(tab_ls)
print("■ 롱온리 스위칭 (개인 실행 가능)"); display(tab_lo)
pd.concat({"long-short": tab_ls, "long-only": tab_lo}).to_csv(f"{RES_DIR}/D1_pair_backtest.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for k, v in cur_ls.items(): ax[0].plot(v.index, (1 + v).cumprod(), lw=1, label=k)
for k, v in cur_lo.items(): ax[1].plot(v.index, (1 + v).cumprod(), lw=1, label=k)
for a_, t_ in zip(ax, ["Long-short (net)", "Long-only switching (net)"]):
    a_.set_title(t_); a_.axhline(1, color="grey", lw=0.6); a_.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{RES_DIR}/fig_D1_pairs.png", bbox_inches="tight"); plt.show()

In [ ]:
best = pairs.iloc[0]
grid = []
for lb in [60, 120, 250]:
    for zi in [1.5, 2.0, 2.5]:
        bt = pair_backtest(best["a"], best["b"], lookback=lb, z_in=zi)
        if bt is None: continue
        m = metrics(bt["net"])
        grid.append({"lookback": lb, "z_entry": zi, "CAGR(%)": m["CAGR(%)"],
                     "Sharpe": m["Sharpe"], "MDD(%)": m["MDD(%)"], "trades": bt.attrs["trades"],
                     "round trips": bt.attrs["round_trips"]})
print(f"■ 민감도: {best['A']} / {best['B']}")
tblD2 = pd.DataFrame(grid).set_index(["lookback", "z_entry"])
display(tblD2)
tblD2.to_csv(f"{RES_DIR}/D2_sensitivity.csv", encoding="utf-8-sig")
bt = pair_backtest(best["a"], best["b"])
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(bt.index, bt["z"], lw=0.8); ax.axhline(Z_ENTRY, color="r", ls="--", lw=0.7)
ax.axhline(-Z_ENTRY, color="r", ls="--", lw=0.7); ax.axhline(0, color="grey", lw=0.6)
ax.set_title(f"Spread z-score: {best['A']} / {best['B']}")
plt.tight_layout(); plt.savefig(f"{RES_DIR}/fig_D2_zscore.png", bbox_inches="tight"); plt.show()

---
# Part E. 지주사 할인 — HD한국조선해양 vs HD현대중공업
조선 섹터에만 있는 구조입니다. 지주사는 상장 자회사 지분을 보유하므로 두 주가는 묶여 움직이고,
그 비율이 벌어졌다 좁혀지는지를 같은 z-score 엔진으로 검정합니다.

`STAKE`에 지분율을 넣으면 NAV 할인율로, 비워두면 단순 상대주가로 계산합니다.
HD현대중공업은 2021년 상장이라 표본이 짧다는 점을 결과 해석에 반드시 반영하세요.

In [ ]:
HOLDCO, SUBSID = "009540", "329180"
STAKE = None      # 예: 0.42 (지주사의 자회사 지분율). None이면 단순 상대주가 사용
SHARES = None     # (지주사 발행주식수, 자회사 발행주식수) 튜플. NAV 계산 시 필요

d = pd.DataFrame({"h": close[HOLDCO], "s": close[SUBSID]}).dropna()
print(f"공통 표본: {d.index.min().date()} ~ {d.index.max().date()} ({len(d)}일)")

if STAKE and SHARES:
    nav = close[SUBSID] * SHARES[1] * STAKE
    mcap = close[HOLDCO] * SHARES[0]
    disc = (1 - mcap / nav).dropna()
    disc.plot(figsize=(10, 3.2), title="Holdco NAV discount")
    plt.axhline(disc.mean(), color="r", ls="--", lw=0.8); plt.tight_layout(); plt.show()
    print(f"평균 할인율 {disc.mean():.1%}, 현재 {disc.iloc[-1]:.1%}, ADF p={adfuller(disc)[1]:.3f}")
else:
    sp = np.log(d["h"]) - np.log(d["s"])
    print(f"상대주가 로그스프레드 ADF p = {adfuller(sp)[1]:.3f}  (0.05 미만이면 평균회귀 근거)")
    print(f"반감기 = {half_life(sp):.1f}일")
    sp.plot(figsize=(10, 3.2), title="log(Holdco) - log(Subsidiary)")
    plt.axhline(sp.mean(), color="r", ls="--", lw=0.8)
    plt.tight_layout(); plt.savefig(f"{RES_DIR}/fig_E1_holdco.png", bbox_inches="tight"); plt.show()

bt_ls = pair_backtest(HOLDCO, SUBSID, long_only=False)
bt_lo = pair_backtest(HOLDCO, SUBSID, long_only=True)
rows = []
for lbl, bt in [("Holdco pair long-short", bt_ls), ("Holdco pair long-only", bt_lo)]:
    if bt is None:
        print(f"{lbl}: 표본 부족으로 실행 불가"); continue
    m = metrics(bt["net"], lbl); m["trades"] = bt.attrs["trades"]; m["round trips"] = bt.attrs["round_trips"]
    rows.append(m)
if rows:
    tblE1 = pd.DataFrame(rows).set_index("strategy")
    display(tblE1)
    tblE1.to_csv(f"{RES_DIR}/E1_holdco_backtest.csv", encoding="utf-8-sig")
print("주의: 표본이 짧고 거래 횟수가 적으면 Sharpe는 참고치로만 보세요")

---
# Part A 강건성 점검 (사후 점검)
원래 Part A 결과는 **그대로 두고**, 테마 강도 정의의 세 가지 약점을 바꿔 가며 결론이 유지되는지 봅니다.
결과가 좋게 나와도 원래 결과를 대체하지 않습니다.

| 버전 | 구성 | 시장 조정 |
|---|---|---|
| **V0 원본** | HD현대중공업·HD현대미포·삼성중공업·한화오션·HJ중공업 (구성 변화 있음) | 종목 − KOSPI |
| **V1 대형 3사** | HD한국조선해양·삼성중공업·한화오션 (전 기간 고정) | 종목 − KOSPI |
| **V2 대형 3사 + 베타** | V1과 동일 | t−1일까지 250거래일 롤링 회귀 잔차 |

60일 창, 80% 관측 요건, expanding 백분위 레짐, 상·하위 20%는 원본과 같습니다. 결과는 `results/robust_A/`에 저장합니다.

In [ ]:
# ===== RA1. 세 버전의 테마 강도 =====
ROB_DIR = os.path.join(RES_DIR, "robust_A"); os.makedirs(ROB_DIR, exist_ok=True)
BIG3 = ["009540", "010140", "042660"]          # HD한국조선해양, 삼성중공업, 한화오션
BETA_WIN = 250
BETA_MINP = int(BETA_WIN * 0.8)                 # 베타 창에도 원본과 같은 80% 관측 요건
VERS = ["V0 original", "V1 big3", "V2 big3 beta-adj"]

def lagged_beta_resid(r, m, win=BETA_WIN, minp=BETA_MINP):
    """t일 잔차 = r_t − (α + β·m_t). α·β는 t−1일까지 win일(종목·KOSPI 둘 다 관측된 날)로 추정."""
    x, y = m.where(r.notna()), r.where(m.notna())
    roll = lambda z: z.rolling(win, min_periods=minp).sum()
    n = x.notna().astype(float).rolling(win, min_periods=minp).sum()
    sx, sy, sxx, sxy = roll(x), roll(y), roll(x * x), roll(x * y)
    b = (sxy - sx * sy / n) / (sxx - sx * sx / n)
    a = (sy - b * sx) / n
    beta, alpha = b.shift(1), a.shift(1)      # t일 값에는 t일 데이터가 들어가지 않음
    return r - (alpha + beta * m), beta

ex_v1 = ret[BIG3].sub(mkt, axis=0)
_res, _bet = {}, {}
for t in BIG3:
    _res[t], _bet[t] = lagged_beta_resid(ret[t], mkt)
ex_v2, betas = pd.DataFrame(_res), pd.DataFrame(_bet)

theme_v0_check = rolling_avg_corr(ex_ret, ROLL_WIN)
print("V0 재현 확인 — 기존 theme와의 최대 차이:", float((theme_v0_check - theme).abs().max()))
THEME = {"V0 original": theme,
         "V1 big3": rolling_avg_corr(ex_v1, ROLL_WIN),
         "V2 big3 beta-adj": rolling_avg_corr(ex_v2, ROLL_WIN)}
RANK = {v: s.expanding(min_periods=250).rank(pct=True) for v, s in THEME.items()}
REGIME = {v: to_regime(r) for v, r in RANK.items()}
print("\n베타(t−1일까지 250일 추정) 요약")
display(betas.describe().T[["count", "mean", "min", "max"]].rename(index=NAME))
pd.DataFrame({**{f"theme {v}": s for v, s in THEME.items()},
              **{f"rank {v}": r for v, r in RANK.items()},
              **{f"regime {v}": g for v, g in REGIME.items()},
              **{f"beta {NAME[t]}": betas[t] for t in BIG3}}).to_csv(f"{ROB_DIR}/RA1_theme_series.csv", encoding="utf-8-sig")

In [ ]:
# ===== RA2. 겹쳐 그리기, 버전별 요약, 급등 시점 =====
fig, ax = plt.subplots(figsize=(11, 4.6))
for v, s in THEME.items():
    ax.plot(s.index, s, lw=1.3 if v == "V0 original" else 0.9, label=v)
ax.axvline(pd.Timestamp("2021-09-17"), color="k", ls=":", lw=0.8, label="HD HHI listing (2021-09-17)")
ax.set_ylabel("avg pairwise corr (60d)"); ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False)
ax.set_title("Theme strength robustness: original (5 stocks) vs big-3 vs big-3 beta-adjusted")
plt.tight_layout(); plt.savefig(f"{ROB_DIR}/fig_A1_robust.png", bbox_inches="tight"); plt.show()

rows = []
for v, s in THEME.items():
    both = pd.concat([s, THEME["V0 original"]], axis=1).dropna()
    rows.append({"version": v, "first valid": s.first_valid_index().date(), "valid days": int(s.notna().sum()),
                 "mean": s.mean(), "min": s.min(), "max": s.max(),
                 "corr with V0": both.corr().iloc[0, 1], "overlap days": len(both),
                 "High days": int((REGIME[v] == "High").sum()), "Low days": int((REGIME[v] == "Low").sum())})
tblRA2 = pd.DataFrame(rows).set_index("version")
display(tblRA2); tblRA2.to_csv(f"{ROB_DIR}/RA2_summary.csv", encoding="utf-8-sig")

# High 레짐 일치도 (V1·V2가 High로 본 날 중 V0도 High인 비율)
agree = {v: ((REGIME[v] == "High") & (REGIME["V0 original"] == "High")).sum() / max((REGIME[v] == "High").sum(), 1)
         for v in VERS[1:]}
print("V1·V2의 High 일 중 V0도 High인 비율:", {k: f"{x:.0%}" for k, x in agree.items()})

def top_peaks_r(s, k=6, gap=60):          # 원본 top_peaks와 같은 규칙 (60거래일 이상 간격)
    s = s.dropna().copy(); picks = []
    while len(picks) < k and len(s):
        d = s.idxmax(); picks.append((d.date(), s[d]))
        p = s.index.get_loc(d); s = s.drop(s.index[max(0, p - gap):p + gap + 1])
    return pd.DataFrame(picks, columns=["date", "theme"])
peaks_r = pd.concat({v: top_peaks_r(s) for v, s in THEME.items()}, axis=1)
display(peaks_r); peaks_r.to_csv(f"{ROB_DIR}/RA3_peaks.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ===== RA3. 구성종목 변화 효과 (HD현대중공업 편입, HD현대미포 제외) =====
MINOBS = int(ROLL_WIN * 0.8)
in_hhi = ret["329180"].rolling(ROLL_WIN).count() >= MINOBS
in_mipo = ret["010620"].rolling(ROLL_WIN).count() >= MINOBS
d_in = in_hhi.idxmax()                      # HD현대중공업이 V0 평균에 처음 들어간 날
d_out = in_mipo[in_mipo].index[-1]          # HD현대미포가 V0 평균에 마지막으로 들어간 날
theme_v0_exhhi = rolling_avg_corr(ex_ret.drop(columns="329180"), ROLL_WIN)   # V0에서 HD현대중공업만 뺀 것
cal = ret.index
typ = THEME["V0 original"].diff().abs().median()
rows = []
for lab, d0, d1 in [("HD HHI enters V0", cal[cal.get_loc(d_in) - 1], d_in),
                    ("HD Mipo leaves V0", d_out, cal[cal.get_loc(d_out) + 1])]:
    rows.append({"change": lab, "day before": d0.date(), "day of": d1.date(),
                 "V0 before": THEME["V0 original"][d0], "V0 after": THEME["V0 original"][d1],
                 "V0 jump": THEME["V0 original"][d1] - THEME["V0 original"][d0],
                 "V0 ex-HHI jump (same day)": theme_v0_exhhi[d1] - theme_v0_exhhi[d0],
                 "V1 jump (same day)": THEME["V1 big3"][d1] - THEME["V1 big3"][d0],
                 "median |daily change| of V0": typ})
tblRA4a = pd.DataFrame(rows).set_index("change"); display(tblRA4a)

i0 = cal.get_loc(d_in)
pre, post = cal[max(0, i0 - 250):i0], cal[i0:i0 + 250]
span = cal[(cal >= d_in) & (cal <= d_out)]
lev = {"V0 original": THEME["V0 original"], "V0 ex-HHI (4 stocks)": theme_v0_exhhi,
       "V1 big3": THEME["V1 big3"], "V2 big3 beta-adj": THEME["V2 big3 beta-adj"]}
tblRA4b = pd.DataFrame({k: {"mean 250d before HHI entry": s.reindex(pre).mean(),
                            "mean 250d after HHI entry": s.reindex(post).mean(),
                            "change": s.reindex(post).mean() - s.reindex(pre).mean()} for k, s in lev.items()}).T
display(tblRA4b)
eff = (THEME["V0 original"] - theme_v0_exhhi).reindex(span)
print(f"HD현대중공업 편입 구간({d_in.date()}~{d_out.date()}) 평균 V0 − V0(HHI 제외) = {eff.mean():+.4f}")
tblRA4a.to_csv(f"{ROB_DIR}/RA4_composition_jumps.csv", encoding="utf-8-sig")
tblRA4b.to_csv(f"{ROB_DIR}/RA4_composition_levels.csv", encoding="utf-8-sig")

In [ ]:
# ===== RA4. A2 재계산: 레짐별 선행 섹터 초과수익 (Newey-West) =====
sector_ex3 = ex_v1.mean(axis=1, skipna=True)                 # 대형 3사 동일가중 초과수익 (종목 − KOSPI)
TARGET = {"V0 original": sector_ex, "V1 big3": sector_ex3, "V2 big3 beta-adj": sector_ex3}
rows = []
for v in VERS:
    reg, tgt = REGIME[v], TARGET[v]
    for h in [5, 20, 60]:
        f = fwd_sum(tgt, h)
        for gname in ["Low", "Mid", "High"]:
            m, t, n = nw_mean(f[reg == gname], h)
            rows.append({"version": v, "horizon": f"{h}d", "regime": gname, "mean(%)": m * 100, "NW t": t, "N": n})
        sub = reg.isin(["Low", "High"])
        dd, t = nw_diff(f[sub], (reg[sub] == "High"), h)
        rows.append({"version": v, "horizon": f"{h}d", "regime": "High − Low", "mean(%)": dd * 100, "NW t": t, "N": np.nan})
longA2 = pd.DataFrame(rows)
tblRA5 = longA2.pivot_table(index=["horizon", "regime"], columns="version", values=["mean(%)", "NW t", "N"], dropna=False)
order = [(h, g) for h in ["5d", "20d", "60d"] for g in ["Low", "Mid", "High", "High − Low"]]
tblRA5 = tblRA5.reindex(order)[[(c, v) for v in VERS for c in ["mean(%)", "NW t", "N"]]]
display(tblRA5); tblRA5.to_csv(f"{ROB_DIR}/RA5_A2_by_version.csv", encoding="utf-8-sig")
print("섹터 수익: V0 = 원본 5종목, V1·V2 = 대형 3사 동일가중 (V1·V2는 레짐 정의만 다름)")

In [ ]:
# ===== RA5. B-H3 재계산: 레짐별 공시 반응 =====
# 이벤트 정의는 원래 보고서(outputs/)와 같게 원공시만 사용. 기존 raw·events 변수는 건드리지 않음
raw_r = raw[raw["report_nm"].astype(str).str.match(r"^단일판매[ㆍ·]\s*공급계약\s*체결")].copy()
print(f"수주 공시: 기존 필터 {len(raw)}건 → 원공시만 {len(raw_r)}건")
events_r = build_events(raw_r, ret.index)
AR_r = run_event_study(events_r, ret, mkt)
ev_r = events_r.loc[AR_r.index].copy()
ev_r["own_pre"], ev_r["own_ann"] = car(AR_r, -10, -1), car(AR_r, 0, 1)
src_r = ev_r.assign(grp=ev_r["ticker"].map(GROUP)).drop_duplicates(["grp", "tdate"])
_pr = [{"src": i, "ticker": p, "tdate": e["tdate"], "pos": e["pos"]}
       for i, e in src_r.iterrows() for p in ret.columns
       if p not in CORR_EXCLUDE and GROUP.get(p) != GROUP[e["ticker"]]]
peer_ev_r = pd.DataFrame(_pr)
AR_pr = run_event_study(peer_ev_r, ret, mkt)
peer_ev_r = peer_ev_r.loc[AR_pr.index]
src_r = src_r.join(pd.DataFrame({"src": peer_ev_r["src"].values, "peer_pre": car(AR_pr, -10, -1).values,
                                 "peer_ann": car(AR_pr, 0, 1).values}).groupby("src").mean(), how="inner")
print(f"이벤트 {len(ev_r)}개, 스필오버 원천 이벤트 {len(src_r)}개 (이벤트·경쟁사 정의는 버전과 무관하게 동일, 레짐만 바뀜)")

rows_m, rows_w = [], []
for v in VERS:
    rl = RANK[v].shift(1)                                     # 공시 전날 기준 레짐 (원본과 동일)
    ev_r["reg"] = to_regime(rl.reindex(ev_r["tdate"])).values
    src_r["reg"] = to_regime(rl.reindex(src_r["tdate"])).values
    for df, cols in [(ev_r, ["own_ann", "own_pre"]), (src_r, ["peer_ann", "peer_pre"])]:
        for c in cols:
            for gname in ["Low", "Mid", "High"]:
                x = df.loc[df["reg"] == gname, c].dropna()
                rows_m.append({"version": v, "variable": c, "regime": gname, "mean(%)": x.mean() * 100, "N": len(x)})
            a, b = df.loc[df["reg"] == "High", c].dropna(), df.loc[df["reg"] == "Low", c].dropna()
            p = stats.ttest_ind(a, b, equal_var=False)[1] if min(len(a), len(b)) >= 3 else np.nan
            rows_w.append({"version": v, "variable": c, "High − Low (%p)": (a.mean() - b.mean()) * 100,
                           "Welch p": p, "N High": len(a), "N Low": len(b)})
tblRA6m = pd.DataFrame(rows_m).pivot_table(index=["variable", "regime"], columns="version", values=["mean(%)", "N"])
tblRA6m = tblRA6m.reindex([(c, g) for c in ["own_ann", "own_pre", "peer_ann", "peer_pre"] for g in ["Low", "Mid", "High"]])
tblRA6m = tblRA6m[[(c, v) for v in VERS for c in ["mean(%)", "N"]]]
tblRA6w = pd.DataFrame(rows_w).pivot_table(index="variable", columns="version",
                                           values=["High − Low (%p)", "Welch p", "N High", "N Low"])
tblRA6w = tblRA6w.reindex(["own_ann", "own_pre", "peer_ann", "peer_pre"])[
    [(c, v) for v in VERS for c in ["High − Low (%p)", "Welch p", "N High", "N Low"]]]
display(tblRA6m); display(tblRA6w)
tblRA6m.to_csv(f"{ROB_DIR}/RA6_H3_means_by_version.csv", encoding="utf-8-sig")
tblRA6w.to_csv(f"{ROB_DIR}/RA6_H3_welch_by_version.csv", encoding="utf-8-sig")

In [ ]:
# ===== RA6. C-S1 재계산: 테마 과열(백분위 ≥ 0.8)이면 비중 0 =====
big3_ret = ret[BIG3].mean(axis=1, skipna=True).fillna(0)        # 대형 3사 동일가중
ASSET = {"V0 original": sector_ret, "V1 big3": big3_ret, "V2 big3 beta-adj": big3_ret}
ones = pd.Series(1.0, index=ret.index)
rows = []
for v in VERS:
    r_ = RANK[v]
    s = (r_ < 0.8).astype(float); s[r_.isna()] = 1.0           # 레짐 판정 불가 구간은 보유 (원본 규칙)
    bt, bh = backtest(s, ASSET[v]), backtest(ones, ASSET[v])
    for lbl, sl in [("Full", slice(None, None)), ("In-sample ~2020", slice(None, IS_END)),
                    ("Out-of-sample 2021~", slice(IS_END, None))]:
        m1, m2 = metrics(bt["net"].loc[sl]), metrics(bh["net"].loc[sl])
        rows.append({"version": v, "period": lbl, "S1 CAGR(%)": m1["CAGR(%)"], "S1 Sharpe": m1["Sharpe"],
                     "S1 MDD(%)": m1["MDD(%)"], "B&H CAGR(%)": m2["CAGR(%)"], "B&H Sharpe": m2["Sharpe"],
                     "B&H MDD(%)": m2["MDD(%)"], "S1 − B&H CAGR(%p)": m1["CAGR(%)"] - m2["CAGR(%)"],
                     "avg weight": bt["w"].loc[sl].mean(),
                     "turnover/yr": bt["w"].loc[sl].diff().abs().sum() / (len(bt.loc[sl]) / 252)})
tblRA7 = pd.DataFrame(rows).set_index(["period", "version"]).reindex(
    [(p, v) for p in ["Full", "In-sample ~2020", "Out-of-sample 2021~"] for v in VERS])
display(tblRA7); tblRA7.to_csv(f"{ROB_DIR}/RA7_S1_by_version.csv", encoding="utf-8-sig")
print("자산: V0 = 원본 5종목 섹터, V1·V2 = 대형 3사 동일가중. 비교는 같은 자산의 B&H 대비(S1 − B&H)로 볼 것")

In [ ]:
# ===== RA7. HD한국조선해양(009540) 수정주가 점검: 2019-06 물적분할, 2017 인적분할 =====
k = "009540"
chk = pd.DataFrame({"close": close[k], "volume": vol[k], "ret(%)": ret[k] * 100, "KOSPI ret(%)": mkt * 100})
print("■ 2019-06-03 물적분할 전후"); display(chk.loc["2019-05-27":"2019-06-12"])
print("■ 2017 인적분할 거래정지(2017-03-30~05-08) 전후"); display(chk.loc["2017-03-27":"2017-05-15"].dropna(subset=["ret(%)"]))
yr = ret[k].abs().groupby(ret.index.year).agg(["max", "idxmax"])
yr["max"] *= 100; display(yr.rename(columns={"max": "max |ret|(%)", "idxmax": "date"}))
chk.loc["2017-03-01":"2019-12-31"].to_csv(f"{ROB_DIR}/RA8_009540_split_check.csv", encoding="utf-8-sig")